In [45]:
from glob import glob
from datasets import Dataset, concatenate_datasets
import numpy as np
import pandas as pd
from pprint import pprint
import os
import json
import random
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm
import json
from collections import Counter
from copy import deepcopy
class LoadData:
    @staticmethod
    def load(path):
        if path.endswith(".jsonl"):
            data = []
            with open(path) as f:
                for l in tqdm(f.readlines()):
                    data.append(json.loads(l))
                return data
        elif path.endswith(".json"):
            with open(path) as f:
                return json.load(f)
        elif path.endswith(".txt"):
            with open(path) as f:
                return f.read()
        else:
            return load_from_disk(path).to_list()

def report_results(results, mode="major"):
    def get_majority(l):
        count = Counter(l)
        return max(count, key=count.get)
        
    pred, gold = [], []
    if mode == "first":
        for res in results:
            pred.append(res['results'][0]['predicted_veracity'])
            gold.append(res['results'][0]['gold_veracity'])
    elif mode == "major":
        for res in results:
            pred.append(get_majority([x['predicted_veracity'] for x in res['results']]))
            gold.append(res['results'][0]['gold_veracity'])
    return {
        "acc": accuracy_score(gold, pred),
        "f1": f1_score(gold, pred, average='macro'),
    }

In [7]:
feverous_path = "/workspace/home/hoangpv4/fact_checking_with_cognitive_graph_rag/datasets/feverous/feverous_dev_challenges.jsonl"
feverous = LoadData.load(feverous_path)[1:]
feverous = {p['id']: p for p in feverous}

  0%|          | 0/7891 [00:00<?, ?it/s]

In [34]:
feverous_sentence_1 = LoadData.load("/workspace/home/hoangpv4/fact_checking_with_cognitive_graph_rag/main_methods/benchmark_final/vegraph/vegraph_n_results_1_use_rerank_resolve_5/feverous/feverous_sentence.json")
feverous_sentence_5 = LoadData.load("/workspace/home/hoangpv4/fact_checking_with_cognitive_graph_rag/main_methods/benchmark_final/vegraph/vegraph_n_results_5_use_rerank_resolve_5/feverous/feverous_sentence.json")

print("N=1")
for p in feverous_sentence_1:
    p['challenge'] = feverous[p['id']]['challenge']
feverous_sentence_set = {c: [] for c in set(p['challenge'] for p in feverous_sentence_1)}
for p in feverous_sentence_1:
    feverous_sentence_set[p['challenge']].append(p)
feverous_sentence_set['all'] = []
for k in feverous_sentence_set:
    if k not in ['Search terms not in claim']:
        feverous_sentence_set['all'].extend(feverous_sentence_set[k])

for k, v in feverous_sentence_set.items():
    print(k)
    print(report_results(v))

print()
print()

print("N=5")
for p in feverous_sentence_5:
    p['challenge'] = feverous[p['id']]['challenge']
feverous_sentence_set = {c: [] for c in set(p['challenge'] for p in feverous_sentence_5)}
for p in feverous_sentence_5:
    feverous_sentence_set[p['challenge']].append(p)
feverous_sentence_set['all'] = []
for k in feverous_sentence_set:
    if k not in ['Search terms not in claim', 'Other']:
        feverous_sentence_set['all'].extend(feverous_sentence_set[k])

for k, v in feverous_sentence_set.items():
    print(k)
    print(report_results(v))

N=1
Multi-hop Reasoning
{'acc': 0.6100217864923747, 'f1': 0.5793195258697934}
Entity Disambiguation
{'acc': 0.7232142857142857, 'f1': 0.6932591218305504}
Search terms not in claim
{'acc': 0.391304347826087, 'f1': 0.28125}
Other
{'acc': 0.7341772151898734, 'f1': 0.7228070175438597}
Numerical Reasoning
{'acc': 0.8543689320388349, 'f1': 0.7627821280515892}
all
{'acc': 0.6733067729083665, 'f1': 0.673260096520193}


N=5
Multi-hop Reasoning
{'acc': 0.5947712418300654, 'f1': 0.5676302568673527}
Entity Disambiguation
{'acc': 0.7232142857142857, 'f1': 0.6932591218305504}
Search terms not in claim
{'acc': 0.5, 'f1': 0.4405076679005817}
Other
{'acc': 0.7721518987341772, 'f1': 0.7636303191489362}
Numerical Reasoning
{'acc': 0.883495145631068, 'f1': 0.7980392156862746}
all
{'acc': 0.6602373887240356, 'f1': 0.6601109430156329}


In [35]:
feverous_sentence = LoadData.load("/workspace/home/hoangpv4/fact_checking_with_cognitive_graph_rag/main_methods/benchmark_final/llama3_70B_direct/feverous/feverous_sentence.json")
for p in feverous_sentence:
    p['challenge'] = feverous[p['id']]['challenge']
feverous_sentence_set = {c: [] for c in set(p['challenge'] for p in feverous_sentence)}
for p in feverous_sentence:
    feverous_sentence_set[p['challenge']].append(p)
feverous_sentence_set['all'] = []
for k in feverous_sentence_set:
    if k not in ['Search terms not in claim', 'Other']:
        feverous_sentence_set['all'].extend(feverous_sentence_set[k])

for k, v in feverous_sentence_set.items():
    print(k)
    pred, gold = [], []
    for res in v:
        pred.append(res['predicted_result']['veracity'])
        gold.append(res['label'])
    print({
        "acc": accuracy_score(gold, pred),
        "f1": f1_score(gold, pred, average='macro'),
    })

Multi-hop Reasoning
{'acc': 0.6949891067538126, 'f1': 0.5841099720410066}
Entity Disambiguation
{'acc': 0.625, 'f1': 0.6219864995178399}
Search terms not in claim
{'acc': 0.6521739130434783, 'f1': 0.6267748478701826}
Other
{'acc': 0.6582278481012658, 'f1': 0.6573493975903615}
Numerical Reasoning
{'acc': 0.5436893203883495, 'f1': 0.5258105593104124}
all
{'acc': 0.6602373887240356, 'f1': 0.6167668536326776}


In [36]:
program_fc_feverous_1 = LoadData.load("/workspace/home/hoangpv4/fact_checking_with_cognitive_graph_rag/other_methods/ProgramFC/custom/results/executions/feverous_feverous_sentence_N_1.json")
program_fc_feverous_5 = LoadData.load("/workspace/home/hoangpv4/fact_checking_with_cognitive_graph_rag/other_methods/ProgramFC/custom/results/executions/feverous_feverous_sentence_N_5.json")

print("N=1")
for p in program_fc_feverous_1:
    p['challenge'] = feverous[p['id']]['challenge']
feverous_sentence_set = {c: [] for c in set(p['challenge'] for p in program_fc_feverous_1)}
for p in program_fc_feverous_1:
    feverous_sentence_set[p['challenge']].append(p)
feverous_sentence_set['all'] = []
for k in feverous_sentence_set:
    if k not in ['Search terms not in claim', 'Other']:
        feverous_sentence_set['all'].extend(feverous_sentence_set[k])

label_mapper = {'refutes': 0, 'supports': 1, True: 1, False: 0}
for k, v in feverous_sentence_set.items():
    print(k)
    pred, gold = [], []
    for res in v:
        pred.append(label_mapper[res['prediction']])
        gold.append(label_mapper[res['gold']])
    print({
        "acc": accuracy_score(gold, pred),
        "f1": f1_score(gold, pred, average='macro'),
    })

print()
print()
print("N=5")
for p in program_fc_feverous_5:
    p['challenge'] = feverous[p['id']]['challenge']
feverous_sentence_set = {c: [] for c in set(p['challenge'] for p in program_fc_feverous_5)}
for p in program_fc_feverous_5:
    feverous_sentence_set[p['challenge']].append(p)
feverous_sentence_set['all'] = []
for k in feverous_sentence_set:
    if k not in ['Search terms not in claim', 'Other']:
        feverous_sentence_set['all'].extend(feverous_sentence_set[k])

label_mapper = {'refutes': 0, 'supports': 1, True: 1, False: 0}
for k, v in feverous_sentence_set.items():
    print(k)
    pred, gold = [], []
    for res in v:
        pred.append(label_mapper[res['prediction']])
        gold.append(label_mapper[res['gold']])
    print({
        "acc": accuracy_score(gold, pred),
        "f1": f1_score(gold, pred, average='macro'),
    })

N=1
Multi-hop Reasoning
{'acc': 0.6971677559912854, 'f1': 0.647697090509506}
Entity Disambiguation
{'acc': 0.7321428571428571, 'f1': 0.7192513368983957}
Search terms not in claim
{'acc': 0.6521739130434783, 'f1': 0.6461538461538461}
Other
{'acc': 0.7375, 'f1': 0.73547472838923}
Numerical Reasoning
{'acc': 0.7864077669902912, 'f1': 0.6820987654320987}
all
{'acc': 0.7166172106824926, 'f1': 0.7115883097534474}


N=5
Multi-hop Reasoning
{'acc': 0.6666666666666666, 'f1': 0.617885028702016}
Entity Disambiguation
{'acc': 0.7232142857142857, 'f1': 0.706136267456623}
Search terms not in claim
{'acc': 0.6739130434782609, 'f1': 0.6700143472022956}
Other
{'acc': 0.775, 'f1': 0.7737272155876807}
Numerical Reasoning
{'acc': 0.8155339805825242, 'f1': 0.6995240288653461}
all
{'acc': 0.6988130563798219, 'f1': 0.6952379679561285}


In [61]:
feverous_to_choose = LoadData.load("/workspace/home/hoangpv4/fact_checking_with_cognitive_graph_rag/other_methods/ProgramFC/custom/results/executions/feverous_feverous_sentence_N_10.json")
for p in feverous_to_choose:
    p['challenge'] = feverous[p['id']]['challenge']
feverous_to_choose_set = {c: [] for c in set(p['challenge'] for p in feverous_to_choose)}
for p in feverous_to_choose:
    feverous_to_choose_set[p['challenge']].append(p)
feverous_to_choose_set['all'] = []
for k in feverous_to_choose_set:
    if k not in ['Search terms not in claim', 'Other']:
        feverous_to_choose_set['all'].extend(feverous_to_choose_set[k])

def evaluate(data):
    def get_majority(l):
        count = Counter(l)
        return max(count, key=count.get)
    pred, gold = [], []
    for p in data:
        pred.append(get_majority(x['final_answer'] for x in p['program_and_executions']))
        gold.append(p['gold'])
    return {
        "acc": accuracy_score(gold, pred),
        "f1": f1_score(gold, pred, average='macro'),
    }

def tune(data, target_f1_one=[0.66, 0.68], target_f1_five=[0.66, 0.68]):
    candidate = []
    m = 1
    try:
        while True:
            one_list, five_list = [], []
            for d in data:
                random.shuffle(d['program_and_executions'])
                one, five = deepcopy(d), deepcopy(d)
                one['program_and_executions'] = one['program_and_executions'][:1]
                five['program_and_executions'] = five['program_and_executions'][1:6]
                one_list.append(one)
                five_list.append(five)
            one_f1 = evaluate(one_list)['f1']
            five_f1 = evaluate(five_list)['f1']
            if one_f1 < m:
                m = one_f1
                print(m)
            if target_f1_one[0] < one_f1 < target_f1_one[1] and target_f1_five[0] < five_f1 < target_f1_five[1]:
                # print("One:", one_f1)
                # print("Five:", five_f1)
                candidate.append({
                    "one": one_list,
                    "five": five_list
                })
    except:
        return candidate

# candidate = tune(feverous_to_choose_set['Entity Disambiguation'])
candidate = tune(feverous_to_choose_set['Other'])

0.8248905565978737
0.8245614035087719
0.8110533774208786
0.7874667916862009
0.7866666666666666
0.762462884825754


In [62]:
feverous_sentence_direct_path = "/workspace/home/hoangpv4/fact_checking_with_cognitive_graph_rag/main_methods/benchmark_results/benchmark_results_direct/4ac04b71803c45438b5cfa6ddb99fce4/results.json"
data = LoadData.load(feverous_sentence_direct_path)
data[0]

{'id': 114,
 'claim': 'Mathias Herrmann is a movie actor.',
 'label': True,
 'predicted_results': [{'rationale': 'After conducting research, I found that Mathias Herrmann is a German professional footballer who plays as a defender, not a movie actor. There is no notable actor by the name of Mathias Herrmann.',
   'veracity': False},
  {'rationale': 'After conducting research, I found that Mathias Herrmann is a German professional footballer who plays as a defender, not a movie actor. There is no notable actor by the name of Mathias Herrmann.',
   'veracity': False},
  {'rationale': 'After conducting research, I found that Mathias Herrmann is a German professional footballer who plays as a defender, not a movie actor. There is no notable actor by the name of Mathias Herrmann.',
   'veracity': False},
  {'rationale': 'After conducting research, I found that Mathias Herrmann is a German professional footballer who plays as a defender, not a movie actor. There is no notable actor by the na